# 04. Low Scorecard Failure Analysis

This notebook complements the trace-focused repair analysis by asking: **where did the final plan fail most according to the scorecards?**

It focuses on low Travel Agent scorecard pass rates and low rationale pass rates, then connects those failures to:

- hard/commonsense constraint verdicts
- rationale verification verdicts
- fragile or missing evidence links
- final TravelPlan structure
- trace behavior where exact query-string matching is available

Important caveat: several traces use slightly different query budgets than the evaluated final plans. Trace joins therefore use exact query string matching and keep unmatched low-score cases visible.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import math
import re
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path('..').resolve()
EVAL_ROOT = ROOT / 'data' / 'evaluation'
PLAN_ROOT = ROOT / 'data' / 'travelplans'
TP_TRACE_PATH = ROOT / 'data' / 'traces' / 'travel_agent_full'
BL_TRACE_PATH = ROOT / 'data' / 'traces' / 'baseline'

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)


In [ ]:
def safe_loads(value: Any) -> Any:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, (dict, list)):
        return value
    try:
        return json.loads(value)
    except Exception:
        return None


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))


def query_id_from_path(path: Path) -> str:
    if path.name in {'scorecard.json', 'hard_constraints.json'}:
        return path.parent.name
    return path.stem


def verdict_counts(items: list[dict[str, Any]], key: str = 'final_verdict') -> dict[str, int]:
    counts = Counter(str(item.get(key) or 'MISSING').upper() for item in items)
    return {
        'checks': len(items),
        'pass': counts.get('PASS', 0),
        'fail': counts.get('FAIL', 0),
        'missing': counts.get('MISSING_INFO', 0) + counts.get('MISSING', 0),
        'na': counts.get('N/A', 0) + counts.get('NA', 0),
    }


def pass_rate(counts: dict[str, int]) -> float:
    denom = counts['pass'] + counts['fail'] + counts['missing']
    return counts['pass'] / denom if denom else np.nan


def plan_slots(plan_obj: dict[str, Any] | None):
    travelplan = (plan_obj or {}).get('travelplan') or plan_obj
    if not isinstance(travelplan, dict):
        return
    for day in travelplan.get('days') or []:
        for pos, slot in enumerate(day.get('slots') or [], start=1):
            yield day, pos, slot


def slot_text(slot: dict[str, Any]) -> str:
    return ' '.join(str(slot.get(k) or '') for k in ['name', 'description', 'location', 'notes'])


def tool_output_text(outputs_json: Any) -> str:
    obj = safe_loads(outputs_json)
    if not isinstance(obj, dict):
        return '' if obj is None else str(obj)
    out = obj.get('output', obj)
    if isinstance(out, dict) and out.get('content') is not None:
        return str(out.get('content'))
    return json.dumps(out, ensure_ascii=False, default=str)


def parse_tool_args(inputs_json: Any) -> dict[str, Any]:
    obj = safe_loads(inputs_json)
    return obj if isinstance(obj, dict) else {}


def canonical(value: Any) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)


def arg_text(args: dict[str, Any]) -> str:
    for key in ['query', 'origin_address', 'destination_address', 'name', 'description']:
        if args.get(key):
            return str(args[key])
    return canonical(args) if args else ''

ERROR_RE = re.compile(r'(?:timeout|timed out|failed|error|exception|rate limit|unavailable|could not|unable)', re.I)
UNCERTAIN_RE = re.compile(r'(?:uncertain|unknown|not confirm|not confirmed|appears|seems|likely|may|might|fallback|if available|check before|verify)', re.I)
FRAGILE_DOMAINS = {'google.com', 'maps.google', 'instagram.com', 'facebook.com', 'tripadvisor.com', 'booking.com', 'expedia.com', 'agoda.com', 'turbopass.com'}
SEARCH_TOOLS = {'search_web', 'search_flights', 'search_hotels', 'search_restaurants', 'search_attractions'}
MUTATION_TOOLS = {'add_slot', 'insert_slot', 'delete_slot', 'add_day', 'remove_day'}


## 1. Load Final Plans and Evaluation Results

In [ ]:
plan_rows = []
for system in ['travel_agent', 'baseline']:
    plan_dir = PLAN_ROOT / system
    for path in sorted(plan_dir.glob('*.json')) + sorted(plan_dir.glob('*.md')):
        qid = query_id_from_path(path)
        if path.suffix == '.json':
            obj = load_json(path)
            query = obj.get('query')
            qtype = obj.get('description', '')
            travelplan = obj.get('travelplan')
            slot_count = sum(1 for _ in plan_slots(travelplan))
            day_count = len((travelplan or {}).get('days') or []) if isinstance(travelplan, dict) else np.nan
        else:
            obj = None
            query = None
            qtype = ''
            slot_count = np.nan
            day_count = np.nan
        plan_rows.append({'system': system, 'query_id': qid, 'query': query, 'query_type': qtype, 'plan_path': str(path.relative_to(ROOT)), 'day_count': day_count, 'slot_count': slot_count})
plans = pd.DataFrame(plan_rows)

summary_rows = []
detail_rows = []
rationale_rows = []
for system in ['travel_agent', 'baseline']:
    eval_dir = EVAL_ROOT / system
    for score_path in sorted(eval_dir.glob('*/scorecard.json')):
        qid = query_id_from_path(score_path)
        score = load_json(score_path)
        constraints = score.get('aggregated_constraints') or []
        score_counts = verdict_counts(constraints, 'final_verdict')
        hard_items = [c for c in constraints if c.get('constraint_type') == 'hard']
        common_items = [c for c in constraints if c.get('constraint_type') != 'hard']
        hard_counts = verdict_counts(hard_items, 'final_verdict')
        common_counts = verdict_counts(common_items, 'final_verdict')
        rationale = score.get('rationale_verifications') or []
        rat_counts = verdict_counts(rationale, 'verdict')
        summary_rows.append({
            'system': system,
            'query_id': qid,
            'scorecard_checks': score_counts['checks'],
            'scorecard_pass': score_counts['pass'],
            'scorecard_fail': score_counts['fail'],
            'scorecard_missing': score_counts['missing'],
            'scorecard_pass_rate': pass_rate(score_counts),
            'hard_pass_rate': pass_rate(hard_counts),
            'hard_fail': hard_counts['fail'],
            'hard_missing': hard_counts['missing'],
            'common_pass_rate': pass_rate(common_counts),
            'common_fail': common_counts['fail'],
            'common_missing': common_counts['missing'],
            'rationale_checks': rat_counts['checks'],
            'rationale_pass': rat_counts['pass'],
            'rationale_fail': rat_counts['fail'],
            'rationale_missing': rat_counts['missing'],
            'rationale_pass_rate': pass_rate(rat_counts),
        })
        for c in constraints:
            detail_rows.append({'system': system, 'query_id': qid, **c})
        for r in rationale:
            urls = r.get('source_urls') or []
            rationale_rows.append({
                'system': system,
                'query_id': qid,
                'day_index': r.get('day_index'),
                'slot_position': r.get('slot_position'),
                'slot_name': r.get('slot_name'),
                'source_type': r.get('source_type'),
                'verdict': r.get('verdict'),
                'reasoning': r.get('reasoning'),
                'source_urls': urls,
                'url_count': len(urls),
                'has_fragile_url': any(any(domain in str(url) for domain in FRAGILE_DOMAINS) for url in urls),
            })

eval_summary = pd.DataFrame(summary_rows)
eval_details = pd.DataFrame(detail_rows)
rationale_details = pd.DataFrame(rationale_rows)

eval_wide = eval_summary.pivot(index='query_id', columns='system')
eval_wide.columns = [f'{system}_{metric}' for metric, system in eval_wide.columns]
eval_wide = eval_wide.reset_index()
query_meta = plans[plans.system == 'travel_agent'][['query_id', 'query', 'query_type', 'day_count', 'slot_count']]
eval_wide = eval_wide.merge(query_meta, on='query_id', how='left')
display(eval_wide.sort_values('travel_agent_scorecard_pass_rate')[['query_id','query_type','travel_agent_scorecard_pass_rate','travel_agent_hard_pass_rate','travel_agent_rationale_pass_rate','day_count','slot_count']])


## 2. Low Scorecard Cases

The table below selects cases with low Travel Agent scorecard pass rate or low rationale pass rate. These are final-answer failures, not just trace-instability cases.

In [ ]:
low_cases = eval_wide.copy()
low_cases['low_score_reason'] = np.select(
    [
        low_cases['travel_agent_scorecard_pass_rate'] <= 0.80,
        low_cases['travel_agent_rationale_pass_rate'] <= 0.45,
        low_cases['travel_agent_hard_pass_rate'] < 1.0,
    ],
    ['low scorecard pass rate', 'low rationale grounding', 'hard-constraint failure'],
    default='borderline',
)
low_cases = low_cases[
    (low_cases['travel_agent_scorecard_pass_rate'] <= 0.86)
    | (low_cases['travel_agent_rationale_pass_rate'] <= 0.45)
    | (low_cases['travel_agent_hard_pass_rate'] < 1.0)
].copy()
low_cases['score_gap_vs_baseline'] = low_cases['travel_agent_scorecard_pass_rate'] - low_cases['baseline_scorecard_pass_rate']
low_cases['rationale_gap_vs_baseline'] = low_cases['travel_agent_rationale_pass_rate'] - low_cases['baseline_rationale_pass_rate']
low_cases = low_cases.sort_values(['travel_agent_scorecard_pass_rate','travel_agent_rationale_pass_rate'])
cols = [
    'query_id','low_score_reason','travel_agent_scorecard_pass_rate','baseline_scorecard_pass_rate','score_gap_vs_baseline',
    'travel_agent_hard_pass_rate','travel_agent_hard_fail','travel_agent_hard_missing',
    'travel_agent_rationale_pass_rate','baseline_rationale_pass_rate','rationale_gap_vs_baseline',
    'travel_agent_rationale_fail','travel_agent_rationale_missing','day_count','slot_count'
]
display(low_cases[cols])


## 3. What Failed in the Scorecard?

This section lists hard and commonsense constraints that failed or had missing information for the low-score cases.

In [ ]:
low_qids = set(low_cases['query_id'])
failed_constraints = eval_details[
    (eval_details.system == 'travel_agent')
    & (eval_details.query_id.isin(low_qids))
    & (~eval_details.final_verdict.fillna('').eq('PASS'))
].copy()
failed_constraints['constraint_preview'] = failed_constraints['constraint_text'].astype(str).str.replace(r'\s+', ' ', regex=True).str.slice(0, 220)
constraint_failure_table = failed_constraints[[
    'query_id','id','constraint_type','final_verdict','pass_count','fail_count','missing_count','constraint_preview'
]].sort_values(['query_id','constraint_type','id'])
display(constraint_failure_table)

constraint_failure_summary = (
    failed_constraints.groupby(['query_id','constraint_type','final_verdict'])
    .size()
    .reset_index(name='count')
    .sort_values(['query_id','constraint_type','final_verdict'])
)
display(constraint_failure_summary)


## 4. Rationale and Evidence Failure Modes

Low scorecard cases are often dominated by `MISSING_INFO` in rationale verification. This usually means the final plan contains links that the judge could not retrieve or that do not provide enough evidence for the slot claims.

In [ ]:
rat_low = rationale_details[(rationale_details.system == 'travel_agent') & (rationale_details.query_id.isin(low_qids))].copy()
rat_fail = rat_low[rat_low['verdict'] != 'PASS'].copy()
rat_fail['reason_preview'] = rat_fail['reasoning'].fillna('').str.replace(r'\s+', ' ', regex=True).str.slice(0, 260)
rat_fail['urls_preview'] = rat_fail['source_urls'].map(lambda xs: ', '.join(map(str, xs[:2])) if isinstance(xs, list) else '')

rationale_failure_summary = (
    rat_low.groupby(['query_id','verdict'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['PASS','FAIL','MISSING_INFO']:
    if col not in rationale_failure_summary.columns:
        rationale_failure_summary[col] = 0
rationale_failure_summary['checks'] = rationale_failure_summary[['PASS','FAIL','MISSING_INFO']].sum(axis=1)
rationale_failure_summary['missing_or_fail_rate'] = ((rationale_failure_summary['FAIL'] + rationale_failure_summary['MISSING_INFO']) / rationale_failure_summary['checks']).round(3)
display(rationale_failure_summary[['query_id','checks','PASS','FAIL','MISSING_INFO','missing_or_fail_rate']].sort_values('missing_or_fail_rate', ascending=False))

source_failure_summary = (
    rat_fail.groupby(['query_id','source_type','verdict'])
    .agg(count=('slot_name','size'), fragile_url_count=('has_fragile_url','sum'), avg_url_count=('url_count','mean'))
    .reset_index()
    .sort_values(['query_id','count'], ascending=[True, False])
)
source_failure_summary['avg_url_count'] = source_failure_summary['avg_url_count'].round(2)
display(source_failure_summary)

display(Markdown('### Example failed rationale checks'))
display(rat_fail[['query_id','day_index','slot_position','slot_name','source_type','verdict','has_fragile_url','urls_preview','reason_preview']].head(30))


## 5. Final Plan Symptoms for Low-Score Cases

This inspects the final TravelPlan objects directly: missing links, fragile domains, uncertainty language, cost, and category mix.

In [ ]:
def final_plan_symptoms(query_id: str) -> dict[str, Any]:
    path = PLAN_ROOT / 'travel_agent' / f'{query_id}.json'
    obj = load_json(path) if path.exists() else {}
    travelplan = obj.get('travelplan') or {}
    hard = obj.get('hard_constraints') or {}
    slots = list(plan_slots(travelplan) or [])
    total_cost = 0.0
    missing_links = 0
    fragile_links = 0
    uncertainty_slots = 0
    category_counts = Counter()
    for _, _, slot in slots:
        category_counts[str(slot.get('category') or 'other')] += 1
        cost = slot.get('cost')
        if isinstance(cost, (int, float)):
            total_cost += float(cost)
        links = slot.get('links') or []
        if not links:
            missing_links += 1
        if any(any(domain in str(url) for domain in FRAGILE_DOMAINS) for url in links):
            fragile_links += 1
        if UNCERTAIN_RE.search(slot_text(slot)):
            uncertainty_slots += 1
    budget = hard.get('budget_amount')
    return {
        'query_id': query_id,
        'days': len(travelplan.get('days') or []),
        'slots': len(slots),
        'computed_total_cost': total_cost,
        'budget_amount': budget,
        'budget_exceeded': bool(isinstance(budget, (int, float)) and total_cost > float(budget)),
        'missing_link_slots': missing_links,
        'fragile_link_slots': fragile_links,
        'uncertainty_language_slots': uncertainty_slots,
        **{f'category_{k}': v for k, v in category_counts.items()},
    }

plan_symptoms = pd.DataFrame([final_plan_symptoms(qid) for qid in low_cases['query_id']])
display(plan_symptoms.sort_values(['budget_exceeded','missing_link_slots','fragile_link_slots'], ascending=False))


## 6. Trace Context for Low-Score Cases

Trace data is available only when the exact evaluated query string appears in the trace manifest. Cases without trace matches are still meaningful final-answer failures; they just cannot be traced fairly without fuzzy matching.

In [ ]:
tp_manifest = pd.read_csv(TP_TRACE_PATH / 'manifest.csv')
tp_runs = pd.read_csv(TP_TRACE_PATH / 'langsmith_runs.csv')
tp_runs['start_time_dt'] = pd.to_datetime(tp_runs['start_time'], errors='coerce', utc=True)
tp_runs['end_time_dt'] = pd.to_datetime(tp_runs['end_time'], errors='coerce', utc=True)
tp_tools = tp_runs[tp_runs['run_type'] == 'tool'].copy()
tp_tools['tool_args'] = tp_tools['inputs_json'].map(parse_tool_args)
tp_tools['query_norm'] = tp_tools['tool_args'].map(arg_text)
tp_tools['output_text'] = tp_tools['outputs_json'].map(tool_output_text)
tp_tools['has_error_language'] = tp_tools['output_text'].str.contains(ERROR_RE, na=False)
tp_tools['has_uncertainty_language'] = tp_tools['output_text'].str.contains(UNCERTAIN_RE, na=False)
tp_tools = tp_tools.merge(tp_manifest[['thread_id','query','travelplan_title','validation_attempts','validation_passed']], on='thread_id', how='left')
query_lookup = plans[plans.system == 'travel_agent'][['query_id','query']].dropna()
tp_tools = tp_tools.merge(query_lookup, on='query', how='left')

trace_summary = (
    tp_tools.groupby('query_id')
    .agg(
        tp_tool_calls=('name','size'),
        tp_search_calls=('name', lambda s: s.isin(SEARCH_TOOLS).sum()),
        tp_mutation_calls=('name', lambda s: s.isin(MUTATION_TOOLS).sum()),
        tp_error_outputs=('has_error_language','sum'),
        tp_uncertainty_outputs=('has_uncertainty_language','sum'),
        validation_attempts=('validation_attempts','max'),
        validation_passed=('validation_passed','first'),
    )
    .reset_index()
)
low_trace = low_cases[['query_id','travel_agent_scorecard_pass_rate','travel_agent_rationale_pass_rate']].merge(trace_summary, on='query_id', how='left')
low_trace['trace_exact_match_available'] = low_trace['tp_tool_calls'].notna()
low_trace = low_trace.fillna({'tp_tool_calls':0,'tp_search_calls':0,'tp_mutation_calls':0,'tp_error_outputs':0,'tp_uncertainty_outputs':0,'validation_attempts':0})
display(low_trace.sort_values(['trace_exact_match_available','tp_mutation_calls'], ascending=[True, False]))


## 7. Strongest Failure Narratives

This table combines final evaluation failures, final-plan symptoms, and trace context. These are good candidates for presentation examples where the system actually failed a lot, not only where it had trace churn.

In [ ]:
failure_narratives = low_cases[[
    'query_id','low_score_reason','travel_agent_scorecard_pass_rate','travel_agent_hard_pass_rate','travel_agent_rationale_pass_rate',
    'travel_agent_scorecard_fail','travel_agent_hard_fail','travel_agent_rationale_fail','travel_agent_rationale_missing'
]].merge(plan_symptoms, on='query_id', how='left').merge(trace_summary, on='query_id', how='left')
failure_narratives['trace_note'] = np.where(failure_narratives['tp_tool_calls'].notna(), 'trace matched', 'no exact trace match')
failure_narratives[['tp_tool_calls','tp_mutation_calls','tp_search_calls','tp_error_outputs','tp_uncertainty_outputs','validation_attempts']] = failure_narratives[['tp_tool_calls','tp_mutation_calls','tp_search_calls','tp_error_outputs','tp_uncertainty_outputs','validation_attempts']].fillna(0)

failure_narratives['failure_story'] = failure_narratives.apply(
    lambda r: (
        'hard constraints failed; ' if r['travel_agent_hard_pass_rate'] < 1 else ''
    ) + (
        f"{int(r['travel_agent_rationale_missing'])} missing rationale checks; " if r['travel_agent_rationale_missing'] else ''
    ) + (
        f"{int(r['missing_link_slots'])} final slots missing links; " if r['missing_link_slots'] else ''
    ) + (
        f"{int(r['fragile_link_slots'])} fragile-link slots; " if r['fragile_link_slots'] else ''
    ) + (
        f"{int(r['tp_mutation_calls'])} mutation calls in matched trace; " if r['tp_mutation_calls'] else ''
    ) + str(r['trace_note']),
    axis=1,
)

presentation_failure_table = failure_narratives.sort_values([
    'travel_agent_scorecard_pass_rate','travel_agent_rationale_pass_rate'
])[[
    'query_id','low_score_reason','travel_agent_scorecard_pass_rate','travel_agent_hard_pass_rate','travel_agent_rationale_pass_rate',
    'missing_link_slots','fragile_link_slots','uncertainty_language_slots','tp_tool_calls','tp_mutation_calls','validation_attempts','failure_story'
]]
display(presentation_failure_table)


## 8. Main Findings

- `query_16` is the clearest low-scorecard failure: Travel Agent scorecard pass rate 44.4%, hard pass rate 40.0%, and several hard constraints failed.
- `query_11` is a strong grounding failure: scorecard pass rate 76.2%, rationale pass rate 32.7%, and many rationale checks are missing evidence.
- `query_9` has low scorecard pass rate despite hard constraints passing, making it useful for showing final quality gaps beyond hard constraints.
- Several low-score cases have no exact trace match because the trace query differs from the evaluated query. Keep those as final-answer failures, not trace-causal claims.
- The dominant low-score failure mode is evidence/rationale fragility: missing retrieved evidence, fragile links, and unverifiable slot claims.
- This complements the validator-repair notebook: trace churn explains process instability, while low-scorecard analysis explains final answer quality failures.